# CFM Volatility Forecasting Challenge — Notebook principal

**Auteur** : Adam Kerouredan  
**Référence méthodologique** : G. Paleologo, *The Elements of Quantitative Investing*, Wiley 2024

## Objectif

Prédire la volatilité réalisée 14h-16h d'actions US à partir de 54 barres intraday du matin (9h30-13h55).

## Résultat final

**LightGBM F (30 features structurées)** — MAPE holdout = **0.2392** (-35.1% vs baseline CFM officielle)

## Table des matières

| Phase | Description |
|-------|-------------|
| **I**    | Ingestion et analyse exploratoire (EDA) |
| **II**   | Feature Engineering |
| **III**  | Évaluation des features (IC, Marchenko-Pastur) |
| **IV**   | Neutralisation cross-sectionnelle |
| **V**    | Préparation à la modélisation |
| **VI**   | Modélisation — Ridge A, HAR-RV, LightGBM F (champion) |
| **VII**  | Validation industrielle du champion |
| **VIII** | Visualisations et observations |
| **IX**   | Synthèse comparative et pistes pour atteindre le top 1 |

---

# PHASE I — Ingestion et Analyse Exploratoire

## 1.1 Chargement des données

Trois fichiers fournis par CFM :
- `training_input.csv` : 636 313 lignes, 111 colonnes (3 meta + 54 vol + 54 retours)
- `Y_train.csv` : TARGET (volatilité réalisée 14h-16h)
- `testing_input.csv` : 635 397 lignes pour la soumission

**Note importante** : la variable `date` est anonymisée et randomisée par CFM. L'ordre chronologique réel inter-journalier est donc inaccessible. Cette contrainte interdit toute validation walk-forward et justifie l'usage d'une stratified K-Fold.

In [ ]:
import sys
sys.path.append("../")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import DataLoader

loader = DataLoader(data_dir="../data/")
x_train, y_train, x_test = loader.load_all()

print(f"x_train : {x_train.shape}")
print(f"y_train : {y_train.shape}")
print(f"x_test  : {x_test.shape}")

## 1.2 EDA générale

Analyse exhaustive : dimensions, valeurs manquantes, distribution de la TARGET, persistance de volatilité, profil intraday.

In [ ]:
from src.eda_analyzer import EDAAnalyzer

eda = EDAAnalyzer(x_train, y_train, output_dir="../outputs/")
eda.run()

## 1.3 Structure des NaN et profil intraday

**Observations clés** :
- 15,7% des lignes contiennent au moins un NaN
- Concentration à 09h30 (28 091 NaN) — microstructure d'auction d'ouverture
- Profil intraday décroissant en "L" (vol pic à 09h30 puis stabilisation) — pas un U classique car données coupées à 13h55

In [ ]:
# NaN par barre temporelle
missing_per_col = x_train[eda.volatility_columns].isnull().sum()
print("5 premières barres:")
print(missing_per_col.head().to_string())
print("\n5 dernières barres:")
print(missing_per_col.tail().to_string())

# Profil intraday
mean_by_bar = x_train[eda.volatility_columns].mean()
print("\nVol moyenne toutes les 6 barres:")
print(mean_by_bar.iloc[::6].to_string())

## 1.4 Distribution de la TARGET — détection des outliers

**Statistiques** :
- Médiane : 0.142
- Skewness : 5.01 (très asymétrique)
- Kurtosis : 59.9 (queues extrêmes)
- Min : 0.000132 (problématique pour MAPE)
- Max : 7.58

**Implication critique** : la TARGET minimale à 0.000132 fait exploser la métrique MAPE. Une erreur absolue de 0.02 sur une vraie vol de 0.05 donne un MAPE de 40%. Cette propriété guidera plusieurs décisions méthodologiques.

In [ ]:
target = y_train["TARGET"]

p995 = np.percentile(target, 99.5)
p999 = np.percentile(target, 99.9)
p100 = target.max()

print(f"99.5e percentile : {p995:.4f}")
print(f"99.9e percentile : {p999:.4f}")
print(f"Maximum          : {p100:.4f}")
print(f"Valeurs > 99.5p  : {(target > p995).sum()}")
print(f"Valeurs > 99.9p  : {(target > p999).sum()}")

---

# PHASE II — Feature Engineering

## 2.1 Construction des 10 features

**Imputation des NaN** : interpolation linéaire intraday + forward-fill + backward-fill, appliquée ligne par ligne sur l'axe temporel des 54 barres. Aucun leakage car aucune information cross-ligne n'est utilisée.

**Features construites** (10 retenues sur 16 initiales) :

| Famille | Features | Hypothèse économique |
|---------|----------|---------------------|
| Niveau global | `vol_mean` | Persistance de volatilité |
| Asymétrie | `vol_mean_minus_median` | Présence de pics isolés |
| Récence | `vol_mean_recent`, `vol_last_bar` | Proximité de la fenêtre cible |
| Tendance | `vol_linear_slope` | Direction du profil intraday |
| Dispersion | `vol_std`, `vol_min` | Stabilité du régime |
| Accélération | `vol_recent_over_mean` | Dynamique récente vs globale |
| Directionnel | `return_n_positive`, `return_n_negative` | Biais directionnel intraday |

**Features supprimées et pourquoi** :
- `vol_median` (r = 0.969 avec `vol_mean`)
- `vol_trend` (r = 0.918 avec `vol_linear_slope`)
- `vol_max` (r = 0.956 avec `vol_std`)
- `vol_range` (r = 0.999 avec `vol_max`)
- `return_direction_bias` (IC = -0.048, pas de signal)
- `return_last_bar` (IC = +0.002, pas de signal)

In [ ]:
from src.feature_engineer import FeatureEngineer

engineer = FeatureEngineer(x_train, x_test)
features_train, features_test = engineer.build()

print(f"features_train : {features_train.shape}")
print(f"features_test  : {features_test.shape}")
print(f"\nFeatures retenues :")
for col in [c for c in features_train.columns if c != "ID"]:
    print(f"  {col}")

## 2.2 Vérifications post-engineering

In [ ]:
# Shapes, NaN résiduels, alignement ID
print("=== NaN RÉSIDUELS ===")
print(f"features_train : {features_train.isnull().sum().sum()}")
print(f"features_test  : {features_test.isnull().sum().sum()}")

print("\n=== ALIGNEMENT ID ===")
aligned = (features_train["ID"].values == y_train["ID"].values).all()
print(f"features_train['ID'] == y_train['ID'] : {aligned}")

print("\n=== STATS DESCRIPTIVES ===")
feature_cols = [c for c in features_train.columns if c != "ID"]
print(features_train[feature_cols].describe().round(4).to_string())

---

# PHASE III — Évaluation des features

## 3.1 Information Coefficient (Pearson, Kendall τ, cross-sectionnel)

**Référence** : Paleologo (2024), Chap. 8 — Information Coefficient.

Trois métriques calculées :
- **IC Pearson** (linéaire)
- **IC Kendall τ** (rangs, robuste outliers)
- **IC cross-sectionnel Spearman par date** avec t-stat sur 2 117 jours

**Résultats clés** : tous les t-stats > 35, signaux statistiquement très robustes.

In [ ]:
from scipy.stats import spearmanr, kendalltau

target_log = np.log(y_train["TARGET"].values)
feature_cols = [c for c in features_train.columns if c != "ID"]

# --- IC Pearson ---
print("=" * 60)
print("IC PEARSON vs log(TARGET)")
print("=" * 60)
ic_pearson = {}
for col in feature_cols:
    r = np.corrcoef(features_train[col], target_log)[0, 1]
    ic_pearson[col] = r

for name, r in sorted(ic_pearson.items(), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {name:<30} : {r:+.4f}")

# --- IC Kendall tau (rangs, robuste outliers) ---
print("\n" + "=" * 60)
print("IC KENDALL TAU vs log(TARGET)")
print("=" * 60)
ic_kendall = {}
for col in feature_cols:
    tau, pval = kendalltau(features_train[col].values, target_log)
    ic_kendall[col] = (tau, pval)

print(f"  {'Feature':<30} {'Kendall tau':>12} {'p-value':>12}")
for name, (tau, pval) in sorted(ic_kendall.items(), key=lambda x: abs(x[1][0]), reverse=True):
    print(f"  {name:<30} {tau:>+12.4f} {pval:>12.2e}")

## 3.2 Matrice Spearman + IC cross-sectionnel

Audit de la multicolinéarité résiduelle entre features pour s'assurer qu'aucune paire critique ne perturbe les modèles linéaires.

In [ ]:
# --- Matrice Spearman entre features ---
print("=" * 60)
print("MATRICE SPEARMAN ENTRE FEATURES")
print("=" * 60)
X = features_train[feature_cols].values

spearman_matrix = np.zeros((len(feature_cols), len(feature_cols)))
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        r, _ = spearmanr(X[:, i], X[:, j])
        spearman_matrix[i, j] = r

df_spearman = pd.DataFrame(
    spearman_matrix, index=feature_cols, columns=feature_cols
)
print(df_spearman.round(3).to_string())

# --- IC cross-sectionnel ---
print("\n" + "=" * 60)
print("IC CROSS-SECTIONNEL MOYEN (Spearman par date)")
print("=" * 60)

features_with_date = features_train.copy()
features_with_date["date"]       = x_train["date"].values
features_with_date["target_log"] = target_log

cross_section_ics = {col: [] for col in feature_cols}
dates = features_with_date["date"].unique()

for date in dates:
    mask   = features_with_date["date"] == date
    subset = features_with_date[mask]
    if len(subset) < 10:
        continue
    for col in feature_cols:
        r, _ = spearmanr(subset[col].values, subset["target_log"].values)
        if not np.isnan(r):
            cross_section_ics[col].append(r)

print(f"  {'Feature':<30} {'IC moyen':>10} {'IC std':>10} {'t-stat':>10}")
for col in feature_cols:
    ics    = np.array(cross_section_ics[col])
    ic_m   = ics.mean()
    ic_s   = ics.std()
    t_stat = ic_m / (ic_s / np.sqrt(len(ics))) if ic_s > 0 else 0
    print(f"  {col:<30} {ic_m:>+10.4f} {ic_s:>10.4f} {t_stat:>+10.2f}")

## 3.3 Diagnostic Marchenko-Pastur — composantes réelles

**Théorie** : la loi de Marchenko-Pastur (1967) donne la distribution théorique des valeurs propres d'une matrice de covariance de bruit pur avec ratio $\gamma = p/n$.

**Application** : 12 features, 636 313 observations → $\gamma \approx 1.9 \times 10^{-5}$, bord supérieur $\lambda_+ = 1.0087$.

**Résultat** : 3 composantes au-dessus du bruit (PC1 = 6.37, PC2 = 1.79, PC3 = 1.20). Ces 3 composantes économiques sont :
- **PC1** (48% variance) : niveau global de volatilité
- **PC2** (17% variance) : dynamique temporelle intraday
- **PC3** (12% variance) : information directionnelle

**Implication** : un modèle Ridge sur PCA 3 composantes est théoriquement justifié.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Ratio dimensions / observations
p     = len(feature_cols)
n     = len(features_train)
gamma = p / n

# Bornes Marchenko-Pastur (variance unitaire après StandardScaler)
sigma2       = 1.0
lambda_plus  = sigma2 * (1 + np.sqrt(gamma)) ** 2
lambda_minus = sigma2 * (1 - np.sqrt(gamma)) ** 2

print(f"p (features)     : {p}")
print(f"n (observations) : {n:,}")
print(f"gamma            : {gamma:.6f}")
print(f"lambda_+         : {lambda_plus:.6f}")
print(f"lambda_-         : {lambda_minus:.6f}")

# Calcul des valeurs propres empiriques
X_scaled    = StandardScaler().fit_transform(features_train[feature_cols])
cov_matrix  = np.cov(X_scaled.T)
eigenvalues = np.linalg.eigvalsh(cov_matrix)[::-1]

print("\nValeurs propres empiriques vs bord MP :")
n_signal = 0
for i, ev in enumerate(eigenvalues):
    flag = "SIGNAL" if ev > lambda_plus else "BRUIT"
    if ev > lambda_plus:
        n_signal += 1
    print(f"  PC{i+1:2d} : {ev:.4f}  → {flag}")

print(f"\n=> {n_signal} composantes au-dessus du bruit (MP)")

---

# PHASE IV — Neutralisation cross-sectionnelle

**Référence** : Paleologo (2024), Chap. 6 — Loadings Generation.

## 4.1 Neutralisation par date

On retire la composante systématique du marché pour vérifier si le signal est idiosyncratique ou capture simplement les régimes de marché.

In [ ]:
from src.neutralizer import Neutralizer

neutralizer = Neutralizer(date_column="date")

# Neutralisation features et target
features_train_neutral = neutralizer.neutralize_features(features_train, x_train)
target_log_series      = pd.Series(np.log(y_train["TARGET"].values))
target_log_neutral     = neutralizer.neutralize_target(target_log_series, x_train)

print("=== APRÈS NEUTRALISATION ===")
print(f"features shape   : {features_train_neutral.shape}")
print(f"target shape     : {target_log_neutral.shape}")
print(f"target mean      : {target_log_neutral.mean():.6f}  (attendu ~ 0)")
print(f"target std       : {target_log_neutral.std():.4f}")

print("\n=== STATS FEATURES NEUTRALISÉES ===")
print(
    features_train_neutral
    .drop(columns=["ID"])
    .describe()
    .round(4)
    .to_string()
)

## 4.2 IC bruts vs IC neutralisés

**Résultat clé** : drop d'IC moyen de ~13% après neutralisation par date. Le signal est majoritairement idiosyncratique, pas un proxy du beta marché. Une seule feature (`vol_recent_over_mean`) capture surtout du beta.

In [ ]:
# IC sur target neutralisée
target_neutral_array = target_log_neutral.values

print(f"  {'Feature':<30} {'IC brut':>10} {'IC neutralisé':>14} {'Δ':>8}")
print("  " + "-" * 65)

for col in feature_cols:
    r_brut    = ic_pearson[col]
    feat_neut = features_train_neutral[col].values
    r_neut    = np.corrcoef(feat_neut, target_neutral_array)[0, 1]
    delta     = r_neut - r_brut
    print(f"  {col:<30} {r_brut:>+10.4f} {r_neut:>+14.4f} {delta:>+8.4f}")

## 4.3 Neutralisation par stock

Test de robustesse complémentaire : si l'IC chute fortement après dé-mean par stock, le signal était dû au niveau structurel des stocks.

**Résultat** : `vol_mean` IC passe de 0.768 à 0.742 (drop de 3.4% seulement). Le signal n'est pas dû au niveau structurel des stocks. Bon signe pour la généralisation.

In [ ]:
# IC après dé-méan par stock
features_with_stock = features_train.copy()
features_with_stock["product_id"] = x_train["product_id"].values
features_with_stock["target_log"] = np.log(y_train["TARGET"].values)

# Dé-méan vol_mean par stock
stock_means = features_with_stock.groupby("product_id")["vol_mean"].transform("mean")
vol_mean_demean_stock = features_with_stock["vol_mean"] - stock_means

# Dé-méan target par stock
target_stock_means = features_with_stock.groupby("product_id")["target_log"].transform("mean")
target_demean_stock = features_with_stock["target_log"] - target_stock_means

# IC
r = np.corrcoef(vol_mean_demean_stock, target_demean_stock)[0, 1]
print(f"IC vol_mean après dé-méan par stock : {r:.4f}")
print(f"IC vol_mean original                : {ic_pearson['vol_mean']:.4f}")

---

# PHASE V — Préparation à la modélisation

## 5.1 Split holdout 15% intouchable

**Référence** : Paleologo (2024), Chap. 4 — Backtesting Protocol.

`StratifiedShuffleSplit` par quartile de TARGET (random_state=42) :
- **85% train** pour entraînement et cross-validation
- **15% holdout** intouché jusqu'à la validation finale

La stratification garantit que les distributions train/holdout sont indistinguables sur tous les régimes de TARGET.

In [ ]:
import importlib
import src.splitter
importlib.reload(src.splitter)
from src.splitter import Splitter

# Création du split
splitter = Splitter(holdout_size=0.15, n_strata=4, random_state=42)
train_idx, holdout_idx = splitter.split(
    features_train, y_train["TARGET"]
)

# Diagnostic
splitter.diagnose(y_train["TARGET"], train_idx, holdout_idx)

# Stockage des indices pour utilisation ultérieure
print(f"\nIndices train   : {len(train_idx):,}")
print(f"Indices holdout : {len(holdout_idx):,}")

## 5.2 Baselines naïves

**Baseline 1** : moyenne arithmétique des 54 barres intraday. C'est la **baseline officielle CFM**.

**Baseline 2** : moyenne géométrique (`exp(mean(log))`) — alternative log-aware.

In [ ]:
import importlib
import src.evaluator
importlib.reload(src.evaluator)
from src.evaluator import Evaluator

# Préparation - on travaille uniquement sur le train (85%)
y_train_target  = y_train["TARGET"].values
y_train_train   = y_train_target[train_idx]
y_train_holdout = y_train_target[holdout_idx]

# Volatilités brutes par barre (pour les baselines)
vol_columns = [c for c in x_train.columns if c.startswith("volatility")]
vol_data    = x_train[vol_columns].values  # imputation déjà faite par FeatureEngineer
                                            # mais x_train est encore brut ici

# On utilise l'imputation propre déjà faite via features_train
vol_data_imputed = (
    x_train[vol_columns]
    .interpolate(method="linear", axis=1)
    .ffill(axis=1)
    .bfill(axis=1)
    .values
)

# ------------------------------------------------------------------
# Baseline 1 — moyenne brute
# ------------------------------------------------------------------
print("=" * 55)
print("BASELINE 1 — moyenne brute des barres du matin")
print("=" * 55)

baseline_1_train_pred = vol_data_imputed[train_idx].mean(axis=1)

evaluator = Evaluator(apply_jensen_correction=False)
mape_baseline_1 = evaluator.mape(y_train_train, baseline_1_train_pred)
print(f"  MAPE train : {mape_baseline_1:.4f}")

# ------------------------------------------------------------------
# Baseline 2 — moyenne en espace log
# ------------------------------------------------------------------
print("\n" + "=" * 55)
print("BASELINE 2 — moyenne en espace log + exp()")
print("=" * 55)

vol_log_train = np.log(vol_data_imputed[train_idx] + 1e-8)
baseline_2_log_pred = vol_log_train.mean(axis=1)
baseline_2_train_pred = np.exp(baseline_2_log_pred)

mape_baseline_2 = evaluator.mape(y_train_train, baseline_2_train_pred)
print(f"  MAPE train : {mape_baseline_2:.4f}")

# ------------------------------------------------------------------
# Diagnostic complet de la meilleure baseline
# ------------------------------------------------------------------
best_baseline_pred = (
    baseline_1_train_pred if mape_baseline_1 < mape_baseline_2
    else baseline_2_train_pred
)
best_baseline_name = (
    "Baseline 1" if mape_baseline_1 < mape_baseline_2
    else "Baseline 2"
)

print("\n" + "=" * 55)
print(f"DIAGNOSTIC — {best_baseline_name}")
print("=" * 55)

diagnostics = evaluator.diagnose_residuals(
    y_train_train, best_baseline_pred
)
evaluator.print_diagnostics(diagnostics)

## 5.3 Baseline 2 avec correction de Jensen

Test de la correction de Jensen sur la moyenne géométrique. Résultat : sur-correction démontrée empiriquement (MAPE = 2.37), confirmant que la variance résiduelle des baselines est trop grande pour cette correction.

In [ ]:
# Baseline 2 AVEC correction de Jensen
log_vol_train     = np.log(vol_data_imputed[train_idx] + 1e-8)
y_log_train_true  = np.log(y_train_train)

# Prédiction en espace log
y_log_pred_baseline2 = log_vol_train.mean(axis=1)

# Variance résiduelle = un seul scalaire estimé sur le train
residuals          = y_log_train_true - y_log_pred_baseline2
sigma2_residuals   = residuals.var()

print(f"Variance résiduelle estimée : {sigma2_residuals:.4f}")

# Prédiction corrigée
baseline_2_jensen_pred = np.exp(
    y_log_pred_baseline2 + sigma2_residuals / 2
)

mape_baseline_2_jensen = evaluator.mape(
    y_train_train, baseline_2_jensen_pred
)

print(f"\nBaseline 1 (mean)            : MAPE = {mape_baseline_1:.4f}")
print(f"Baseline 2 (logmean)         : MAPE = {mape_baseline_2:.4f}")
print(f"Baseline 2 + Jensen corrigé  : MAPE = {mape_baseline_2_jensen:.4f}")

---

# PHASE VI — Modélisation

## Approche méthodologique

Six modèles ont été testés en cross-validation Repeated Stratified K-Fold (5 folds × 2 répétitions), mais seuls les trois plus pertinents sont conservés dans ce notebook :

1. **Ridge A** — modèle linéaire de référence
2. **HAR-RV** — benchmark académique de Corsi (2009)
3. **LightGBM F** — champion final

Les variantes intermédiaires (Ridge B sur PCA, LightGBM C/D/E, stacking) ont été testées puis écartées. Leurs résultats numériques sont synthétisés en Phase IX pour justifier la trajectoire de modélisation.

## 6.1 Ridge A — Modèle linéaire de référence

**Configuration finale** :
- 10 features brutes
- StandardScaler + winsorisation à 99.5p
- α optimisé par grille log-espacée [0.001, ..., 1000]

**Découverte importante** : MAPE strictement plat sur 6 ordres de grandeur d'α. Le modèle est en **régime sur-échantillonné** (540K obs / 10 features). La régularisation n'a pas d'effet pratique.

**Résultat** : MAPE CV = 0.3090 (-16.8% vs baseline).

In [ ]:
import importlib
import src.model
import src.validator
import src.evaluator
importlib.reload(src.model)
importlib.reload(src.validator)
importlib.reload(src.evaluator)
from src.model import make_ridge_train_fn, predict_fn
from src.validator import Validator
from src.evaluator import Evaluator

# ----------------------------------------------------------------
# Préparation des données train (85%)
# ----------------------------------------------------------------
feature_cols = [c for c in features_train.columns if c != "ID"]

X_full        = features_train[feature_cols].values
y_log_full    = np.log(y_train["TARGET"].values)
y_orig_full   = y_train["TARGET"].values

X_train_data  = X_full[train_idx]
y_log_train   = y_log_full[train_idx]
y_orig_train  = y_orig_full[train_idx]

print(f"Données pour CV : {X_train_data.shape}")

# ----------------------------------------------------------------
# Instanciation Validator + Evaluator
# ----------------------------------------------------------------
validator = Validator(
    n_splits=5, n_repeats=2, n_strata=4, base_random_state=42
)
evaluator = Evaluator(apply_jensen_correction=True)

# ----------------------------------------------------------------
# Configuration des 4 variantes
# ----------------------------------------------------------------
variants = {
    "V1 - Standard sans winso"   : {"scaler_type": "standard", "winsorize": False},
    "V2 - Robust sans winso"     : {"scaler_type": "robust",   "winsorize": False},
    "V3 - Standard + winso 99.5" : {"scaler_type": "standard", "winsorize": True},
    "V4 - Robust + winso 99.5"   : {"scaler_type": "robust",   "winsorize": True},
}

alpha_test = 1.0
results_variants = {}

for variant_name, config in variants.items():
    print(f"\n{'=' * 60}")
    print(f"{variant_name} | alpha = {alpha_test}")
    print(f"{'=' * 60}")

    train_fn = make_ridge_train_fn(alpha=alpha_test, **config)

    cv_results = validator.cross_validate(
        X          = X_train_data,
        y_log      = y_log_train,
        y_original = y_orig_train,
        train_fn   = train_fn,
        predict_fn = predict_fn,
        evaluator  = evaluator,
    )

    validator.print_summary(cv_results)
    results_variants[variant_name] = cv_results

# ----------------------------------------------------------------
# Synthèse comparative
# ----------------------------------------------------------------
print("\n" + "=" * 60)
print("SYNTHÈSE — 4 variantes Ridge A")
print("=" * 60)
print(f"  {'Variante':<35} {'MAPE moyen':>12} {'MAPE std':>10}")
print(f"  {'-'*35} {'-'*12} {'-'*10}")
for name, res in results_variants.items():
    print(f"  {name:<35} {res['mape_mean']:>12.4f} {res['mape_std']:>10.4f}")

print(f"\n  Baseline 1 (référence)              : {mape_baseline_1:.4f}")

In [ ]:
# Grille d'alpha log-espacée
alpha_grid = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

results_alpha = {}

for alpha in alpha_grid:
    print(f"\n{'=' * 60}")
    print(f"Ridge A V3 — alpha = {alpha}")
    print(f"{'=' * 60}")

    train_fn = make_ridge_train_fn(
        alpha=alpha,
        scaler_type="standard",
        winsorize=True,
        winsorize_upper=99.5,
    )

    cv_results = validator.cross_validate(
        X          = X_train_data,
        y_log      = y_log_train,
        y_original = y_orig_train,
        train_fn   = train_fn,
        predict_fn = predict_fn,
        evaluator  = evaluator,
    )

    validator.print_summary(cv_results)
    results_alpha[alpha] = cv_results

# Synthèse
print("\n" + "=" * 60)
print("OPTIMISATION ALPHA — Ridge A V3")
print("=" * 60)
print(f"  {'Alpha':>10} | {'MAPE moyen':>12} | {'MAPE std':>10}")
print(f"  {'-'*10} | {'-'*12} | {'-'*10}")
for alpha, res in results_alpha.items():
    print(f"  {alpha:>10.4f} | {res['mape_mean']:>12.4f} | {res['mape_std']:>10.4f}")

best_alpha = min(results_alpha, key=lambda a: results_alpha[a]['mape_mean'])
print(f"\nMeilleur alpha : {best_alpha}")
print(f"MAPE optimal   : {results_alpha[best_alpha]['mape_mean']:.4f}")
print(f"Baseline 1     : {mape_baseline_1:.4f}")
print(f"Amélioration   : {(mape_baseline_1 - results_alpha[best_alpha]['mape_mean']) / mape_baseline_1:.2%}")

## 6.2 HAR-RV — Benchmark académique (Corsi 2009)

**Référence** : Corsi, F. (2009), *A Simple Approximate Long-Memory Model of Realized Volatility*, Journal of Financial Econometrics.

**Spécification adaptée** :

$$\log(\sigma^{target}) = \beta_0 + \beta_s \log(\bar{\sigma}^{short}) + \beta_m \log(\bar{\sigma}^{medium}) + \beta_l \log(\bar{\sigma}^{long}) + \varepsilon$$

Avec trois horizons emboîtés :
- **Short** : 30 dernières minutes (13h25-13h55)
- **Medium** : 90 dernières minutes (12h25-13h55)
- **Long** : toute la matinée (09h30-13h55)

**Résultat** : MAPE CV = 0.2732 (-26.4% vs baseline)

**Observation clé** : les coefficients estimés donnent un poids de 59% au LONG et seulement 3% au SHORT. C'est l'opposé de l'intuition naïve — c'est le **régime global de la séance** qui prédit, pas les 30 dernières minutes.

In [ ]:
import importlib
import src.model
importlib.reload(src.model)
from src.model import make_ridge_train_fn, predict_fn

# ----------------------------------------------------------------
# HAR-RV — 3 features sur horizons emboîtés
# ----------------------------------------------------------------
print("=" * 60)
print("HAR-RV — Heterogeneous Autoregressive Realized Volatility")
print("Référence : Corsi (2009)")
print("=" * 60)

vol_cols    = [c for c in x_train.columns if c.startswith("volatility")]
vol_data    = x_train[vol_cols].copy()

# Imputation pour cohérence (HAR linéaire ne gère pas NaN)
vol_data_imp = (
    vol_data.interpolate(method="linear", axis=1)
            .ffill(axis=1)
            .bfill(axis=1)
            .values
)

# Trois horizons emboîtés (en partant de la fin = 13h55)
# 54 barres au total, indexées 0 (09h30) à 53 (13h55)
har_short  = vol_data_imp[:, -6:].mean(axis=1)    # 13h25-13h55  (6 barres = 30 min)
har_medium = vol_data_imp[:, -18:].mean(axis=1)   # 12h25-13h55  (18 barres = 90 min)
har_long   = vol_data_imp[:, :].mean(axis=1)      # 09h30-13h55  (54 barres)

# Construction de la matrice de features HAR
X_har_full = np.column_stack([har_short, har_medium, har_long])

# Log-transformation (modèle Corsi original)
X_har_log_full = np.log(X_har_full + 1e-8)

# Sélection train (85%)
X_har_train = X_har_log_full[train_idx]

print(f"\n  Shape X train     : {X_har_train.shape}")
print(f"  Features          : log(short), log(medium), log(long)")
print(f"  Short  = 30 min   : barres 49-54 (13h25-13h55)")
print(f"  Medium = 90 min   : barres 37-54 (12h25-13h55)")
print(f"  Long   = 4h25     : barres 1-54  (09h30-13h55)")

# ----------------------------------------------------------------
# Cross-validation avec Ridge (alpha=0.001 ≈ OLS)
# Winsorisation à 99.5p pour cohérence avec Ridge A
# ----------------------------------------------------------------
train_fn_har = make_ridge_train_fn(
    alpha           = 0.001,
    scaler_type     = "standard",
    winsorize       = True,
    winsorize_upper = 99.5,
)

cv_results_har = validator.cross_validate(
    X          = X_har_train,
    y_log      = y_log_train,
    y_original = y_orig_train,
    train_fn   = train_fn_har,
    predict_fn = predict_fn,
    evaluator  = evaluator,
)

validator.print_summary(cv_results_har)

# ----------------------------------------------------------------
# Inspection des coefficients HAR
# ----------------------------------------------------------------
from src.model import RidgeModel

inspection_har = RidgeModel(
    alpha           = 0.001,
    scaler_type     = "standard",
    winsorize       = True,
    winsorize_upper = 99.5,
)
inspection_har.fit(X_har_train, y_log_train)

har_coefs = inspection_har.ridge.coef_
har_inter = inspection_har.ridge.intercept_

print("\n" + "=" * 60)
print("COEFFICIENTS HAR-RV (après StandardScaler)")
print("=" * 60)
print(f"  beta_short  (30 min)  : {har_coefs[0]:+.4f}")
print(f"  beta_medium (90 min)  : {har_coefs[1]:+.4f}")
print(f"  beta_long   (4h25)    : {har_coefs[2]:+.4f}")
print(f"  intercept             : {har_inter:+.4f}")

# Interprétation
print("\n  Interprétation économique :")
total = abs(har_coefs[0]) + abs(har_coefs[1]) + abs(har_coefs[2])
print(f"    Poids short  : {abs(har_coefs[0])/total*100:.1f}%")
print(f"    Poids medium : {abs(har_coefs[1])/total*100:.1f}%")
print(f"    Poids long   : {abs(har_coefs[2])/total*100:.1f}%")

# ----------------------------------------------------------------
# Tableau de bord complet
# ----------------------------------------------------------------
print("\n" + "=" * 60)
print("TABLEAU DE BORD COMPLET")
print("=" * 60)
print(f"  Baseline 1                  : {mape_baseline_1:.4f}")
print(f"  Ridge A (10 features)       : 0.3090")
print(f"  Ridge B (PCA 3 comps)       : 0.3365")
print(f"  HAR-RV (3 horizons)         : {cv_results_har['mape_mean']:.4f}")

## 6.3 LightGBM F — CHAMPION FINAL

### Pourquoi cette architecture

**Référence** : Paleologo (2024), Chap. 6 — Loadings Generation.

LightGBM C (10 features) atteint MAPE = 0.2600. LightGBM D (108 barres brutes) descend à 0.2584. LightGBM E (+ product_id catégoriel) à 0.2574. Le gain est marginal car LightGBM apprend déjà implicitement l'identité du stock via les niveaux absolus de volatilité.

**LightGBM F** restructure la matrice de features selon le framework de Paleologo : pour chaque feature brute, on construit trois versions complémentaires :

```
Niveau 1 — Feature brute                    (10 features)
Niveau 2 — Z-score cross-sectionnel par date  (10 features)
Niveau 3 — Z-score historique par stock       (10 features)
                                            ─────────────
                                              30 features
```

**Question répondue par chaque niveau** :
- Niveau 1 : *"Quelle est la valeur absolue de cette feature ?"*
- Niveau 2 : *"Ce stock est-il plus volatile que la moyenne du marché aujourd'hui ?"*
- Niveau 3 : *"Ce stock est-il plus volatile que sa norme historique ?"*

**Discipline anti-leakage** : les statistiques par stock (Niveau 3) sont fittées uniquement sur le train. Les Z-scores par date (Niveau 2) n'utilisent jamais la TARGET, donc autorisés sur tout l'échantillon.

### Résultat

**MAPE CV** = 0.2427 (-34.6% vs baseline)  
**MAPE holdout** = 0.2392 (-35.1% vs baseline CFM)

In [ ]:
import importlib
import src.feature_engineer
importlib.reload(src.feature_engineer)
from src.feature_engineer import FeatureTransformer
from src.model import make_lightgbm_train_fn, predict_fn

print("=" * 60)
print("LightGBM F — Features structurées (Paleologo Chap. 6)")
print("=" * 60)

# ----------------------------------------------------------------
# 1. Préparation des features et meta
# ----------------------------------------------------------------
feature_cols = [c for c in features_train.columns if c != "ID"]
meta_full    = x_train[["date", "product_id"]].reset_index(drop=True)

# ----------------------------------------------------------------
# 2. Fit du FeatureTransformer UNIQUEMENT sur le train (85%)
# ----------------------------------------------------------------
transformer = FeatureTransformer(feature_cols=feature_cols)
transformer.fit_stock_stats(
    features_train.iloc[train_idx],
    meta_full.iloc[train_idx],
)

# ----------------------------------------------------------------
# 3. Construction de la matrice complète (brut + Z-date + Z-stock)
# ----------------------------------------------------------------
X_structured_full = transformer.build_full_matrix(
    features_train,
    meta_full,
)
X_structured_train = X_structured_full.iloc[train_idx].values

print(f"\n  Features brutes        : {len(feature_cols)}")
print(f"  Features Z-date        : {len(feature_cols)} (cross-sectionnel)")
print(f"  Features Z-stock       : {len(feature_cols)} (historique par stock)")
print(f"  Shape matrice train    : {X_structured_train.shape}")
print(f"  Total features         : {X_structured_train.shape[1]}")

# ----------------------------------------------------------------
# 4. Cross-validation LightGBM
# ----------------------------------------------------------------
train_fn_f = make_lightgbm_train_fn(
    n_estimators           = 1000,
    max_depth              = 5,
    num_leaves             = 31,
    learning_rate          = 0.03,
    min_child_samples      = 50,
    reg_lambda             = 1.0,
    early_stopping_rounds  = 30,
    internal_val_size      = 0.15,
    random_state           = 42,
)

cv_results_f = validator.cross_validate(
    X          = X_structured_train,
    y_log      = y_log_train,
    y_original = y_orig_train,
    train_fn   = train_fn_f,
    predict_fn = predict_fn,
    evaluator  = evaluator,
)

validator.print_summary(cv_results_f)

# ----------------------------------------------------------------
# 5. Feature importance sur les 30 features structurées
# ----------------------------------------------------------------
from src.model import LightGBMModel

inspection_f = LightGBMModel(
    n_estimators           = 1000,
    max_depth              = 5,
    num_leaves             = 31,
    learning_rate          = 0.03,
    min_child_samples      = 50,
    early_stopping_rounds  = 30,
    random_state           = 42,
)
inspection_f.fit(X_structured_train, y_log_train)

feature_names_f = (
    feature_cols
    + [f"{c}_zd" for c in feature_cols]
    + [f"{c}_zs" for c in feature_cols]
)

print("\n" + "=" * 60)
print("TOP 15 FEATURE IMPORTANCE — LightGBM F")
print("=" * 60)
importance_f = inspection_f.get_feature_importance(
    feature_names   = feature_names_f,
    importance_type = "gain",
)
print(importance_f.head(15).to_string(index=False))

# Importance par famille
total = importance_f["importance"].sum()
brut_imp   = importance_f[~importance_f["feature"].str.endswith(("_zd", "_zs"))]["importance"].sum()
zd_imp     = importance_f[importance_f["feature"].str.endswith("_zd")]["importance"].sum()
zs_imp     = importance_f[importance_f["feature"].str.endswith("_zs")]["importance"].sum()

print(f"\n  Importance par famille :")
print(f"    Features brutes       : {brut_imp/total*100:.1f}%")
print(f"    Z-score date          : {zd_imp/total*100:.1f}%")
print(f"    Z-score stock         : {zs_imp/total*100:.1f}%")

# ----------------------------------------------------------------
# 6. Comparaison globale
# ----------------------------------------------------------------
print("\n" + "=" * 60)
print("TABLEAU DE BORD MIS À JOUR")
print("=" * 60)
# Récapitulatif (valeurs CV issues des exécutions précédentes des variantes éliminées)
print(f"  Baseline 1                       : {mape_baseline_1:.4f}")
print(f"  Ridge A (10 features)            : 0.3090   (exécution précédente)")
print(f"  HAR-RV (3 horizons)              : 0.2732   (exécution précédente)")
print(f"  LightGBM C (10 features)         : 0.2600   (variante écartée)")
print(f"  LightGBM D (108 raw bars)        : 0.2584   (variante écartée)")
print(f"  LightGBM E (+ product_id cat)    : 0.2574   (variante écartée)")
print(f"  LightGBM F (30 structurées)      : {cv_results_f['mape_mean']:.4f}   CHAMPION")

gain_vs_e = (0.2574 - cv_results_f['mape_mean']) / 0.2574
print(f"\n  Gain LightGBM F vs E             : {gain_vs_e:+.2%}")

---

# PHASE VII — Validation industrielle du champion

Audit professionnel complet inspiré des standards quant et des recommandations de Paleologo Chap. 4-5.

## 7.1 Validation finale + bootstrap par date

**Discipline** : le holdout 15% n'a JAMAIS été touché jusqu'ici. Le modèle est entraîné une seule fois sur les 85% train, puis évalué sur le holdout.

**Bootstrap par date** (et non par ligne) pour respecter l'auto-corrélation intra-journalière des observations. 1000 itérations.

In [ ]:
import importlib
import src.feature_engineer
import src.model
importlib.reload(src.feature_engineer)
importlib.reload(src.model)
from src.feature_engineer import FeatureTransformer
from src.model import LightGBMModel

print("=" * 70)
print("VALIDATION INDUSTRIELLE — LightGBM F sur holdout 15%")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Préparation matrice structurée (30 features Paleologo)
# ----------------------------------------------------------------
feature_cols = [c for c in features_train.columns if c != "ID"]
meta_full    = x_train[["date", "product_id"]].reset_index(drop=True)

transformer = FeatureTransformer(feature_cols=feature_cols)
transformer.fit_stock_stats(
    features_train.iloc[train_idx],
    meta_full.iloc[train_idx],
)

X_structured_full   = transformer.build_full_matrix(features_train, meta_full).values
X_structured_train  = X_structured_full[train_idx]
X_structured_holdout = X_structured_full[holdout_idx]

# Cibles alignées avec le split train/holdout
y_orig_train_full = y_train["TARGET"].values[train_idx]
y_orig_holdout    = y_train["TARGET"].values[holdout_idx]
y_log_train_full  = np.log(y_orig_train_full)
y_log_holdout     = np.log(y_orig_holdout)

print(f"\n  Train (85%)   : {X_structured_train.shape}")
print(f"  Holdout (15%) : {X_structured_holdout.shape}")
print(f"  Features      : 10 brutes + 10 Z-date + 10 Z-stock = 30")

# ----------------------------------------------------------------
# 2. Entraînement LightGBM F sur 540K train avec n_est=2000
#    (test piliers 2.4 de Critique)
# ----------------------------------------------------------------
print("\n  Entraînement LightGBM F (n_estimators=2000)...")

final_model_f = LightGBMModel(
    n_estimators           = 2000,    # augmenté vs 1000
    max_depth              = 5,
    num_leaves             = 31,
    learning_rate          = 0.03,
    min_child_samples      = 50,
    reg_lambda             = 1.0,
    early_stopping_rounds  = 50,      # augmenté pour plus de patience
    random_state           = 42,
)
final_model_f.fit(X_structured_train, y_log_train_full)
print(f"  Best iteration : {final_model_f.best_iteration_}")

if final_model_f.best_iteration_ >= 1950:
    print(f"  ⚠ Plafond atteint, considérer n_estimators plus grand")

# ----------------------------------------------------------------
# 3. Variance résiduelle OUT-OF-FOLD pour Jensen propre (pilier 2.5)
#    On utilise la CV pour estimer une variance honnête
# ----------------------------------------------------------------
# Approximation pratique : variance résiduelle de la CV LightGBM F déjà calculée
sigma2_resid_oof = 0.0854  # valeur CV LightGBM F (out-of-fold)
print(f"\n  Variance résiduelle OOF (depuis CV) : {sigma2_resid_oof:.4f}")
print(f"  → utilisée pour Jensen (pas la variance in-sample)")

# ----------------------------------------------------------------
# 4. Prédiction sur holdout
# ----------------------------------------------------------------
y_log_pred_holdout_f = final_model_f.predict(X_structured_holdout)
y_pred_holdout_orig_f = np.exp(y_log_pred_holdout_f + sigma2_resid_oof / 2)

rel_errors_f = np.abs(y_orig_holdout - y_pred_holdout_orig_f) / y_orig_holdout
mape_holdout_f = rel_errors_f.mean()

print(f"\n  MAPE holdout point estimate : {mape_holdout_f:.4f}")

# ----------------------------------------------------------------
# 5. BOOTSTRAP PAR DATE (pilier 2.3 — correction du bug du précédent)
# ----------------------------------------------------------------
print(f"\n  Bootstrap PAR DATE (vraie incertitude)...")

meta_holdout = meta_full.iloc[holdout_idx].reset_index(drop=True)
unique_dates = meta_holdout["date"].unique()
n_unique_dates = len(unique_dates)
print(f"  Dates uniques dans holdout : {n_unique_dates}")

n_bootstrap = 1000
rng         = np.random.default_rng(seed=42)
mape_boot_f = np.empty(n_bootstrap)

# Pré-calcul du mapping date → indices
date_to_indices = {
    d: np.where(meta_holdout["date"].values == d)[0]
    for d in unique_dates
}

for b in range(n_bootstrap):
    # Tirer n_unique_dates dates avec remise
    sampled_dates = rng.choice(unique_dates, size=n_unique_dates, replace=True)
    # Reconstruire les indices d'observations
    indices_boot = np.concatenate([date_to_indices[d] for d in sampled_dates])
    mape_boot_f[b] = rel_errors_f[indices_boot].mean()

ci_lower_f  = np.percentile(mape_boot_f, 2.5)
ci_upper_f  = np.percentile(mape_boot_f, 97.5)
ci_width_f  = ci_upper_f - ci_lower_f

print(f"  Bootstrap par date 1000× :")
print(f"    Moyenne     : {mape_boot_f.mean():.4f}")
print(f"    IC 95%      : [{ci_lower_f:.4f}, {ci_upper_f:.4f}]")
print(f"    Largeur IC  : {ci_width_f:.4f}")

# Comparaison avec bootstrap ligne-par-ligne (ancien LightGBM D)
print(f"\n  Comparaison incertitude :")
print(f"    Bootstrap ligne-par-ligne (LightGBM D) : largeur IC = 0.0112")
print(f"    Bootstrap par date (LightGBM F)        : largeur IC = {ci_width_f:.4f}")
if ci_width_f > 0.015:
    print(f"    → l'IC par date est PLUS LARGE, confirmant l'auto-corrélation intra-date")

## 7.2 Baselines sur holdout + diagnostic post-holdout sur n_estimators

Évaluation des baselines naïves sur le **même holdout** pour une comparaison apples-to-apples avec le champion LightGBM F.

**Test additionnel : `n_estimators = 3000` (diagnostic exploratoire)**

Ce test étend la borne supérieure d'`n_estimators` au-delà de 2000 pour vérifier si le modèle est limité par cet hyperparamètre. Comme il est exécuté après observation du holdout, **ce test est un diagnostic exploratoire post-holdout ; il ne doit pas être interprété comme une sélection strictement out-of-sample**. Le modèle officiellement retenu reste LightGBM F avec `n_estimators = 2000`.

In [ ]:
# ============================================================
# PILIER 2.2 — Baselines évaluées sur le MÊME holdout que LightGBM F
# ============================================================
print("=" * 70)
print("Baselines sur holdout 15% — comparaison apples-to-apples")
print("=" * 70)

# Volatilités intraday du holdout (imputées)
vol_data_holdout = vol_data_imputed[holdout_idx]

# Baseline 1 : moyenne arithmétique des 54 barres (baseline CFM officielle)
pred_b1 = vol_data_holdout.mean(axis=1)
mape_b1_holdout = (np.abs(y_orig_holdout - pred_b1) / y_orig_holdout).mean()

# Baseline 2 : dernière barre 13h55
pred_b2 = vol_data_holdout[:, -1]
mape_b2_holdout = (np.abs(y_orig_holdout - pred_b2) / y_orig_holdout).mean()

# Baseline 3 : moyenne des 6 dernières barres (30 min)
pred_b3 = vol_data_holdout[:, -6:].mean(axis=1)
mape_b3_holdout = (np.abs(y_orig_holdout - pred_b3) / y_orig_holdout).mean()

# Baseline 4 : moyenne des 12 dernières barres (1h)
pred_b4 = vol_data_holdout[:, -12:].mean(axis=1)
mape_b4_holdout = (np.abs(y_orig_holdout - pred_b4) / y_orig_holdout).mean()

print(f"\n  {'Baseline':<35} {'MAPE holdout':>12}")
print(f"  {'-'*35} {'-'*12}")
print(f"  Baseline 1 (moyenne 54 barres, CFM)   {mape_b1_holdout:>12.4f}")
print(f"  Baseline 2 (dernière barre 13h55)     {mape_b2_holdout:>12.4f}")
print(f"  Baseline 3 (moy. 30 dernières min)    {mape_b3_holdout:>12.4f}")
print(f"  Baseline 4 (moy. 1h dernières)        {mape_b4_holdout:>12.4f}")
print(f"  LightGBM F (n_estimators=2000)        {mape_holdout_f:>12.4f}")

# ============================================================
# Diagnostic exploratoire post-holdout : n_estimators = 3000
# ============================================================
print("\n" + "=" * 70)
print("DIAGNOSTIC EXPLORATOIRE — n_estimators = 3000")
print("=" * 70)
print("ATTENTION : ce test est exécuté APRÈS observation du holdout.")
print("Il ne doit pas être interprété comme une sélection strictement out-of-sample.")
print("Le modèle officiel retenu reste LightGBM F (n_estimators = 2000).")

final_model_f_3000 = LightGBMModel(
    n_estimators           = 3000,
    max_depth              = 5,
    num_leaves             = 31,
    learning_rate          = 0.03,
    min_child_samples      = 50,
    reg_lambda             = 1.0,
    early_stopping_rounds  = 80,
    random_state           = 42,
)
final_model_f_3000.fit(X_structured_train, y_log_train_full)

print(f"\n  Best iteration : {final_model_f_3000.best_iteration_}")

y_log_pred_3000 = final_model_f_3000.predict(X_structured_holdout)
y_pred_3000     = np.exp(y_log_pred_3000 + sigma2_resid_oof / 2)
mape_3000       = (np.abs(y_orig_holdout - y_pred_3000) / y_orig_holdout).mean()

print(f"  MAPE holdout (n_est=3000)     : {mape_3000:.4f}")
print(f"  MAPE holdout (n_est=2000)     : {mape_holdout_f:.4f}")
print(f"  Gain marginal post-holdout    : {(mape_holdout_f - mape_3000)/mape_holdout_f:+.2%}")
print(f"\n  Note : ce gain est dans la marge de bruit de l'IC bootstrap")
print(f"  (largeur IC = {ci_upper_f - ci_lower_f:.4f}, gain = {mape_holdout_f - mape_3000:.4f})")

## 7.3 Audit complet : Top-vol, Underestimation, Pairwise, QLIKE

**Pilier 5 — Audit top-vol** : MAPE par déciles extrêmes (10%, 5%, 1%, 0.5%, 0.1%) pour vérifier la performance en queue.

**Pilier 6 — Underestimation analysis** : critique pour le risk management. Quantifie le biais de sous-estimation par décile TARGET.

**Pilier 7 — Pairwise comparison** : test statistique observation par observation entre LightGBM D et F, avec bootstrap par date sur la différence.

**Pilier 8 — QLIKE** (Patton-Sheppard 2009, Paleologo Chap. 5) : métrique rank-robust pour les forecasts de volatilité, asymétrique (pénalise plus la sous-estimation que la sur-estimation).

In [ ]:
# ============================================================
# PILIER 5 — AUDIT TOP-VOL (déciles extrêmes)
# Évaluation du champion LightGBM F (n_estimators = 2000)
# ============================================================
print("=" * 70)
print("PILIER 5 — Audit top-vol (queue de distribution)")
print("LightGBM F champion (n_estimators = 2000)")
print("=" * 70)

# Prédictions du champion strict (LightGBM F n_est=2000, calculées en 7.1)
rel_errors_final = rel_errors_f
y_pred_final     = y_pred_holdout_orig_f

# Buckets de queue de distribution
buckets = [
    ("Bottom 90% (calmes)",   0,     0.90),
    ("Top 10%",                0.90,  1.00),
    ("Top 5%",                 0.95,  1.00),
    ("Top 1%",                 0.99,  1.00),
    ("Top 0.5%",               0.995, 1.00),
    ("Top 0.1%",               0.999, 1.00),
]

print(f"\n  {'Bucket':<25} {'TARGET range':<30} {'MAPE':>8} {'Obs':>8}")
print(f"  {'-'*25} {'-'*30} {'-'*8} {'-'*8}")

for name, lo, hi in buckets:
    if lo == 0:
        mask = y_orig_holdout <= np.percentile(y_orig_holdout, hi * 100)
    else:
        lo_val = np.percentile(y_orig_holdout, lo * 100)
        hi_val = np.percentile(y_orig_holdout, hi * 100) if hi < 1.0 else y_orig_holdout.max()
        mask = (y_orig_holdout > lo_val) & (y_orig_holdout <= hi_val)
    if mask.sum() == 0:
        continue
    targets_b = y_orig_holdout[mask]
    mape_b    = rel_errors_final[mask].mean()
    print(
        f"  {name:<25} "
        f"[{targets_b.min():.4f} - {targets_b.max():.4f}]   "
        f"{mape_b:>8.4f} "
        f"{mask.sum():>8,}"
    )

# ============================================================
# PILIER 6 — UNDERESTIMATION ANALYSIS
# Critique pour risk management : où le modèle sous-estime systématiquement ?
# ============================================================
print("\n" + "=" * 70)
print("PILIER 6 — Underestimation analysis (critique pour risk mgmt)")
print("=" * 70)

underestimation_mask  = y_pred_final < y_orig_holdout
underestimation_amt   = np.where(
    underestimation_mask,
    (y_orig_holdout - y_pred_final),
    0,
)
relative_under = np.where(
    underestimation_mask,
    underestimation_amt / y_orig_holdout,
    0,
)
bias_pct = (y_pred_final / y_orig_holdout) - 1

print(f"\n  Statistiques globales :")
print(f"    % observations underestimées        : {underestimation_mask.mean():.2%}")
print(f"    Mean underestimation (absolue)      : {underestimation_amt[underestimation_mask].mean():.4f}")
print(f"    Mean underestimation (relative)     : {relative_under[underestimation_mask].mean():.4f}")
print(f"    Bias moyen (pred/true - 1)          : {bias_pct.mean():+.4f}")

# Underestimation par décile TARGET
print(f"\n  Underestimation par décile TARGET (critique pour risk) :")
print(f"    {'Décile':<10} {'TARGET range':<25} {'% under':>10} {'Mean under rel':>15}")
for d in range(10):
    lo = d * 10
    hi = (d + 1) * 10
    lo_val = np.percentile(y_orig_holdout, lo)
    hi_val = np.percentile(y_orig_holdout, hi) if hi < 100 else y_orig_holdout.max()
    mask = (y_orig_holdout >= lo_val) & (y_orig_holdout < hi_val)
    if d == 9:
        mask = y_orig_holdout >= lo_val
    if mask.sum() == 0:
        continue
    pct_under = underestimation_mask[mask].mean()
    mean_under = (
        relative_under[mask & underestimation_mask].mean()
        if (mask & underestimation_mask).sum() > 0 else 0
    )
    print(
        f"    D{d+1:<8} "
        f"[{lo_val:.4f} - {hi_val:.4f}]   "
        f"{pct_under:>10.2%} "
        f"{mean_under:>15.4f}"
    )

# ============================================================
# PILIER 7 — PAIRWISE COMPARISON (référence LightGBM D pour démonstrer le gain de F)
# ============================================================
print("\n" + "=" * 70)
print("PILIER 7 — Pairwise comparison (référence : LightGBM D entraîné séparément)")
print("=" * 70)

# Entraînement LightGBM D pour la comparaison
final_model_d = LightGBMModel(
    n_estimators           = 1000,
    max_depth              = 5,
    num_leaves             = 31,
    learning_rate          = 0.03,
    min_child_samples      = 50,
    reg_lambda             = 1.0,
    early_stopping_rounds  = 30,
    random_state           = 42,
)
final_model_d.fit(X_raw_full[train_idx], y_log_train_full)
y_log_pred_d = final_model_d.predict(X_raw_full[holdout_idx])

# Variance résiduelle OOF pour LightGBM D (estimée précédemment)
sigma2_resid_d = 0.0941
y_pred_d     = np.exp(y_log_pred_d + sigma2_resid_d / 2)
rel_errors_d = np.abs(y_orig_holdout - y_pred_d) / y_orig_holdout

mape_d_holdout = rel_errors_d.mean()
print(f"\n  MAPE LightGBM D sur holdout : {mape_d_holdout:.4f}")
print(f"  MAPE LightGBM F sur holdout : {mape_holdout_f:.4f}")

# Test pairwise
diff_error = rel_errors_d - rel_errors_final
print(f"\n  Distribution de la différence d'erreur (D - F) :")
print(f"    Mean        : {diff_error.mean():+.6f}")
print(f"    % obs où F bat D   : {(diff_error > 0).mean():.2%}")
print(f"    % obs où D bat F   : {(diff_error < 0).mean():.2%}")

# Bootstrap par date sur la différence
n_bootstrap = 1000
diff_boot   = np.empty(n_bootstrap)
for b in range(n_bootstrap):
    sampled_dates = rng.choice(unique_dates, size=len(unique_dates), replace=True)
    indices_boot = np.concatenate([date_to_indices_holdout[d] for d in sampled_dates])
    diff_boot[b] = diff_error[indices_boot].mean()

ci_lower_diff = np.percentile(diff_boot, 2.5)
ci_upper_diff = np.percentile(diff_boot, 97.5)
print(f"\n  IC 95% bootstrap par date sur (MAPE_D - MAPE_F) :")
print(f"    [{ci_lower_diff:+.6f}, {ci_upper_diff:+.6f}]")
if ci_lower_diff > 0:
    print(f"    F domine D statistiquement (IC strictement positif)")
else:
    print(f"    Pas de domination statistique claire")

# ============================================================
# PILIER 8 — QLIKE loss (Paleologo Chap. 5, Patton-Sheppard 2009)
# ============================================================
print("\n" + "=" * 70)
print("PILIER 8 — QLIKE loss (Paleologo Chap. 5)")
print("=" * 70)
print("  Définition : QLIKE = (1/T) Σ [σ²_true/σ²_pred - log(σ²_true/σ²_pred) - 1]")
print("  Asymétrique : pénalise plus la sous-estimation que la sur-estimation")

def qlike(y_true, y_pred, eps=1e-8):
    ratio = (y_true ** 2 + eps) / (y_pred ** 2 + eps)
    return np.mean(ratio - np.log(ratio) - 1)

qlike_d  = qlike(y_orig_holdout, y_pred_d)
qlike_f  = qlike(y_orig_holdout, y_pred_final)
qlike_b1 = qlike(y_orig_holdout, pred_b1)
qlike_b4 = qlike(y_orig_holdout, pred_b4)

print(f"\n  {'Modèle':<35} {'QLIKE':>10} {'MAPE':>10}")
print(f"  {'-'*35} {'-'*10} {'-'*10}")
print(f"  Baseline 1 (moy 54 barres)            {qlike_b1:>10.4f} {mape_b1_holdout:>10.4f}")
print(f"  Baseline 4 (moy 1h)                   {qlike_b4:>10.4f} {mape_b4_holdout:>10.4f}")
print(f"  LightGBM D                            {qlike_d:>10.4f} {mape_d_holdout:>10.4f}")
print(f"  LightGBM F (CHAMPION)                 {qlike_f:>10.4f} {mape_holdout_f:>10.4f}")

print(f"\n  Cohérence des classements :")
all_models = [qlike_b1, qlike_b4, qlike_d, qlike_f]
print(f"    QLIKE classe F en premier : {'OUI' if qlike_f == min(all_models) else 'NON'}")
print(f"    MAPE classe F en premier  : OUI (par construction)")
print(f"    -> Les deux métriques s'accordent sur le choix du champion")

## 7.4 Métriques complètes (MAPE, RMSE, MAE, R²)

Métriques de régression standard pour caractériser la performance du modèle au-delà du MAPE seul.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("=" * 70)
print("MÉTRIQUES COMPLÈTES — Comparaison D vs F (champion) sur holdout 15%")
print("=" * 70)

# LightGBM D — déjà calculé en Pilier 7
mape_d_metric = mape_d_holdout
rmse_d = np.sqrt(mean_squared_error(y_orig_holdout, y_pred_d))
mae_d  = mean_absolute_error(y_orig_holdout, y_pred_d)
r2_d   = r2_score(y_orig_holdout, y_pred_d)

# LightGBM F champion (n_est=2000)
mape_f_metric = mape_holdout_f
rmse_f = np.sqrt(mean_squared_error(y_orig_holdout, y_pred_holdout_orig_f))
mae_f  = mean_absolute_error(y_orig_holdout, y_pred_holdout_orig_f)
r2_f   = r2_score(y_orig_holdout, y_pred_holdout_orig_f)

print(f"\n  {'Métrique':<10} {'LightGBM D':>12} {'LightGBM F':>12} {'Gain (F vs D)':>15}")
print(f"  {'-'*10} {'-'*12} {'-'*12} {'-'*15}")
print(f"  {'MAPE':<10} {mape_d_metric:>12.4f} {mape_f_metric:>12.4f} {(mape_d_metric - mape_f_metric)/mape_d_metric:>+15.2%}")
print(f"  {'RMSE':<10} {rmse_d:>12.4f} {rmse_f:>12.4f} {(rmse_d - rmse_f)/rmse_d:>+15.2%}")
print(f"  {'MAE':<10} {mae_d:>12.4f} {mae_f:>12.4f} {(mae_d - mae_f)/mae_d:>+15.2%}")
print(f"  {'R²':<10} {r2_d:>12.4f} {r2_f:>12.4f} {(r2_f - r2_d):>+15.4f}")

print(f"\n  Interprétation :")
print(f"    Ratio RMSE/MAE LightGBM F  : {rmse_f/mae_f:.2f}  (gaussien attendu = 1.25)")
print(f"    -> Ratio plus élevé que le gaussien : queues plus épaisses (kurtosis +)")
print(f"\n    R² LightGBM F  : {r2_f*100:.1f}% de variance expliquée")
print(f"    R² LightGBM D  : {r2_d*100:.1f}%")
print(f"    Gain absolu en R² : +{(r2_f - r2_d)*100:.1f} points")

---

# PHASE VIII — Visualisations et observations

Trois groupes de visualisations pour comprendre la performance du modèle au-delà des chiffres agrégés.

## 8.1 Diagnostic 4 panneaux

Génère quatre graphes :
1. **Predicted vs Actual** : calibration globale du modèle
2. **Résidus vs prédictions** : détection d'hétéroscédasticité
3. **Distribution des erreurs relatives** : présence d'outliers
4. **QQ-plot des résidus log** : normalité de l'hypothèse log

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.stats as stats

sns.set_style("whitegrid")
plt.rcParams["font.size"] = 10

# Variables du champion LightGBM F (calculées en Phase VII)
# y_orig_holdout, y_pred_holdout_orig_f, y_log_pred_holdout_f, y_log_holdout

residuals_log_f  = y_log_holdout - y_log_pred_holdout_f
residuals_orig_f = y_orig_holdout - y_pred_holdout_orig_f
rel_errors_f     = np.abs(residuals_orig_f) / y_orig_holdout

# ============================================================
# FIGURE 1 — Diagnostic global du modèle LightGBM F (4 panneaux)
# ============================================================
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 2, hspace=0.3, wspace=0.25)

# --- 1. Predicted vs Actual ---
ax1 = fig.add_subplot(gs[0, 0])
n_plot     = min(10000, len(y_orig_holdout))
idx_sample = np.random.RandomState(42).choice(
    len(y_orig_holdout), n_plot, replace=False
)
ax1.scatter(
    y_orig_holdout[idx_sample], y_pred_holdout_orig_f[idx_sample],
    alpha=0.3, s=8, color="#2c7fb8", edgecolors="none",
)
max_val = min(y_orig_holdout.max(), y_pred_holdout_orig_f.max(), 2.0)
ax1.plot([0, max_val], [0, max_val], "r--", linewidth=1.5, label="Identité y=x")
ax1.set_xlabel("Vraie volatilité (TARGET)")
ax1.set_ylabel("Volatilité prédite")
ax1.set_title("Predicted vs Actual (LightGBM F sur holdout)", fontweight="bold")
ax1.set_xlim(0, max_val)
ax1.set_ylim(0, max_val)
ax1.legend()

# --- 2. Résidus vs prédictions ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(
    y_pred_holdout_orig_f[idx_sample], residuals_orig_f[idx_sample],
    alpha=0.3, s=8, color="#41b6c4", edgecolors="none",
)
ax2.axhline(0, color="red", linestyle="--", linewidth=1.5)
ax2.set_xlabel("Prédiction")
ax2.set_ylabel("Résidu (vrai - prédit)")
ax2.set_title("Résidus vs prédiction", fontweight="bold")
ax2.set_xlim(0, 1.5)
ax2.set_ylim(-1, 1)

# --- 3. Distribution des erreurs relatives ---
ax3 = fig.add_subplot(gs[1, 0])
rel_errors_clipped = np.clip(rel_errors_f, 0, 2)
ax3.hist(
    rel_errors_clipped, bins=80,
    color="#7fcdbb", edgecolor="black", linewidth=0.5, alpha=0.7,
)
ax3.axvline(
    rel_errors_f.mean(), color="red",
    linestyle="--", linewidth=2,
    label=f"MAPE moyen = {rel_errors_f.mean():.4f}",
)
ax3.axvline(
    np.median(rel_errors_f), color="orange",
    linestyle="--", linewidth=2,
    label=f"Médian = {np.median(rel_errors_f):.4f}",
)
ax3.set_xlabel("Erreur relative")
ax3.set_ylabel("Nombre d'observations")
ax3.set_title("Distribution des erreurs relatives", fontweight="bold")
ax3.legend()

# --- 4. QQ-plot des résidus log ---
ax4 = fig.add_subplot(gs[1, 1])
stats.probplot(residuals_log_f, dist="norm", plot=ax4)
ax4.set_title("QQ-plot des résidus log vs loi normale", fontweight="bold")
ax4.get_lines()[0].set_markerfacecolor("#2c7fb8")
ax4.get_lines()[0].set_markersize(3)
ax4.get_lines()[0].set_alpha(0.5)
ax4.get_lines()[1].set_color("red")
ax4.get_lines()[1].set_linewidth(1.5)

plt.suptitle(
    "Diagnostic LightGBM F (champion) sur holdout 15%",
    fontsize=14, fontweight="bold", y=0.995,
)
plt.savefig("../outputs/diagnostic_lightgbm_f.png", dpi=150, bbox_inches="tight")
plt.show()

### Observations sur le diagnostic 4 panneaux

**Panneau 1 — Predicted vs Actual**  
Le nuage de points suit globalement la diagonale y=x, confirmant une calibration globale correcte. Le nuage s'évase vers les grandes valeurs — c'est l'hétéroscédasticité attendue : plus la vraie vol est élevée, plus la dispersion des prédictions augmente. Quelques outliers visibles en haut (cas où le modèle prédit 1.7 alors que la vraie valeur est ~0.2).

**Panneau 2 — Résidus vs prédiction**  
Nuage centré autour de zéro, sans pattern systématique. Pas de biais directionnel. L'évasement vers la droite confirme l'hétéroscédasticité.

**Panneau 3 — Distribution des erreurs relatives**  
Pic près de 0 avec une décroissance rapide. La majorité des prédictions ont une erreur < 25%. Médiane (0.19) < Moyenne (0.26), donc le MAPE est tiré vers le haut par des queues longues.

**Panneau 4 — QQ-plot des résidus log**  
Le centre suit la ligne théorique gaussienne. Les queues s'éloignent — distribution leptokurtique (kurtosis +2.7) avec queues plus épaisses que la gaussienne. Cohérent avec la finance — événements extrêmes plus fréquents que sous une gaussienne.

## 8.2 MAPE par quartile de TARGET

In [ ]:
# Visualisation MAPE par quartile pour LightGBM F (champion)
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

quartiles_h    = pd.qcut(y_orig_holdout, q=4, labels=False, duplicates="drop")
quartile_names = ["Q1 (calmes)", "Q2", "Q3", "Q4 (volatils)"]
mape_q         = []
for q in range(4):
    mask = quartiles_h == q
    rel_err_q = np.abs(y_orig_holdout[mask] - y_pred_holdout_orig_f[mask]) / y_orig_holdout[mask]
    mape_q.append(rel_err_q.mean())

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#d73027", "#fc8d59", "#fee090", "#91bfdb"]
bars   = ax.bar(quartile_names, mape_q, color=colors, edgecolor="black")

for bar, val in zip(bars, mape_q):
    ax.text(
        bar.get_x() + bar.get_width() / 2, val + 0.005,
        f"{val:.4f}",
        ha="center", va="bottom", fontweight="bold",
    )

mape_global = np.mean(np.abs(y_orig_holdout - y_pred_holdout_orig_f) / y_orig_holdout)
ax.axhline(mape_global, color="black", linestyle="--", linewidth=1.5,
           label=f"MAPE global = {mape_global:.4f}")
ax.set_ylabel("MAPE")
ax.set_title("MAPE par quartile de TARGET — LightGBM F (champion)", fontweight="bold")
ax.legend()
ax.set_ylim(0, max(mape_q) * 1.15)
plt.tight_layout()
plt.savefig("../outputs/mape_by_quartile_lightgbm_f.png", dpi=150, bbox_inches="tight")
plt.show()

### Observations sur MAPE par quartile

Le MAPE n'est pas distribué uniformément sur les régimes de volatilité :

- **Q1 (jours calmes, TARGET < 0.10)** : MAPE ≈ 36% — c'est ce quartile qui tire le MAPE global vers le haut
- **Q2, Q3 (jours moyens)** : MAPE ≈ 20% — performance solide
- **Q4 (jours volatils, TARGET > 0.21)** : MAPE ≈ 21% — performance maintenue sur les régimes de stress

**Cette répartition est exactement celle qu'on souhaite pour un modèle de risk management** : le modèle prédit mieux les jours volatils que les jours calmes en relatif. La dégradation sur Q1 est structurelle à la métrique MAPE (très sensible aux petites valeurs) et non pas un défaut de modélisation.

**Pour la défense orale** : *"Sur les jours où le risk management compte vraiment — les régimes de forte volatilité — le modèle atteint 21% d'erreur relative, contre 36% sur les jours calmes peu informatifs. Le MAPE global de 24% est tiré par Q1 mais cette dégradation est métrique-induite, pas modèle-induite."*

## 8.3 Exemples de prédictions intraday

In [ ]:
# Six exemples de prédictions LightGBM F : 2 calmes, 2 modérés, 2 volatils
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

sorted_idx       = np.argsort(y_orig_holdout)
n_holdout        = len(y_orig_holdout)
calm_idx         = sorted_idx[100:102]
medium_idx       = sorted_idx[n_holdout // 2 - 1:n_holdout // 2 + 1]
volatile_idx     = sorted_idx[-102:-100]

selected_idx     = np.concatenate([calm_idx, medium_idx, volatile_idx])
holdout_indices  = holdout_idx
bar_times        = np.arange(54)

for i, idx_in_holdout in enumerate(selected_idx):
    ax            = axes[i // 3, i % 3]
    orig_idx      = holdout_indices[idx_in_holdout]

    vol_intraday  = vol_data_imputed[orig_idx]
    target_actual = y_orig_holdout[idx_in_holdout]
    target_pred   = y_pred_holdout_orig_f[idx_in_holdout]

    ax.plot(bar_times, vol_intraday, marker="o", markersize=3, linewidth=1,
            color="#2c7fb8", label="Vol intraday (9h30-13h55)")
    ax.axhline(target_actual, color="green", linestyle="--", linewidth=2,
               label=f"Vraie vol 14h-16h = {target_actual:.4f}")
    ax.axhline(target_pred, color="red", linestyle=":", linewidth=2,
               label=f"Prédite (F) = {target_pred:.4f}")

    err_rel = abs(target_actual - target_pred) / target_actual
    regime  = "Calme" if i < 2 else "Modéré" if i < 4 else "Volatile"
    ax.set_title(f"{regime} — MAPE = {err_rel:.2%}", fontweight="bold")
    ax.set_xlabel("Barre intraday (5 min)")
    ax.set_ylabel("Volatilité")
    ax.legend(fontsize=8)

plt.suptitle("Exemples de prédictions LightGBM F — 6 cas couvrant le spectre de régimes",
             fontsize=14, fontweight="bold", y=1.0)
plt.tight_layout()
plt.savefig("../outputs/examples_predictions_lightgbm_f.png", dpi=150, bbox_inches="tight")
plt.show()

### Observations sur les exemples intraday

**Cas calmes (Q1)** — MAPE 200%+  
La vol intraday présente quelques pics ponctuels (0.3-0.4) mais la vraie vol après-midi est très faible (~0.025). Le modèle est "trompé" par les pics et prédit ~0.08 alors que la vraie valeur est ~0.025. **Erreur absolue petite (0.05) mais erreur relative énorme**. C'est exactement le problème Q1 que la métrique MAPE amplifie structurellement.

**Cas modérés (Q2-Q3)** — MAPE 5-23%  
Performance excellente. Le modèle capture bien la dynamique intraday et prédit avec moins de 25% d'erreur. Sur les jours typiques, le modèle est très précis.

**Cas volatils (Q4)** — MAPE 14-18%  
Sur les jours de forte volatilité (TARGET > 1.5), le modèle capture correctement le régime de stress. Il sous-estime légèrement les pics extrêmes mais prédit dans le bon ordre de grandeur. **C'est précisément ce qu'on veut pour un risk model : être fiable quand la vol est élevée**.

---

# PHASE IX — Synthèse comparative et pistes futures

## 9.1 Tableau récapitulatif des modèles testés

Huit variantes ont été évaluées au total. Le tableau ci-dessous résume chaque variante avec sa justification méthodologique et la décision retenue.

| Modèle | MAPE (CV) | Justification du test | Décision |
|--------|----------:|----------------------|----------|
| **Baseline 1** (mean 54 barres) | 0.3713 | Baseline CFM officielle (persistance naïve) | Référence |
| **Baseline 4** (mean 1h) | — | Test d'une moyenne plus locale | Battue par tous les modèles supervisés |
| **Ridge A** (10 features) | 0.3090 | Modèle linéaire de référence avec winsorisation | Conservé (référence linéaire) |
| **Ridge B** (PCA 3 comps) | 0.3365 | Test de la réduction PCA Marchenko-Pastur | Écarté : perte de 24% de variance utile |
| **HAR-RV** (3 horizons) | 0.2732 | Benchmark académique Corsi (2009) | Conservé (benchmark) |
| **LightGBM C** (10 features) | 0.2600 | Test des non-linéarités sur features engineered | Écarté : moins bon que F |
| **LightGBM D** (108 barres) | 0.2584 | Test de l'information séquentielle brute | Écarté : gain marginal |
| **LightGBM E** (+ product_id) | 0.2574 | Test ajout identité stock catégorielle | Écarté : gain marginal |
| **Stacking** (HAR-RV + LGBM) | 0.2614 | Test residual stacking académique | Écarté : erreurs corrélées |
| **LightGBM F** (30 features structurées) | **0.2427** | Framework Paleologo Chap. 6 — Loadings Generation | **CHAMPION** |

## 9.2 Justification du choix final

**LightGBM F** est retenu pour quatre raisons :

1. **Performance** : -34.6% vs baseline CFM en cross-validation, -34.9% sur holdout intouchable (n_estimators=2000)
2. **Justification théorique** : framework Paleologo Chap. 6 (Z-scores cross-sectionnels et historiques)
3. **Interprétabilité** : 30 features avec hypothèse économique claire (vs 108 barres anonymes)
4. **Comportement risk-aware** : MAPE plus faible sur Q4 (0.2083) que sur Q1 (0.3573) — profil souhaité pour un modèle de risque

**Pourquoi pas les variantes intermédiaires** :

- **Ridge B (PCA)** : la PCA Marchenko-Pastur retrouve les 3 dimensions économiques construites manuellement mais perd 24% de variance utile.
- **LightGBM C/D/E** : trois variantes successives avec gains marginaux (<1% chacune). LightGBM apprenait déjà implicitement l'identité du stock via les niveaux absolus de volatilité.
- **Stacking HAR-RV + LightGBM** : les deux modèles utilisent essentiellement les mêmes inputs (volatilité matin). Leurs erreurs sont corrélées, l'ensemble n'apporte pas de gain.

## 9.3 Limitations identifiées

Trois limitations ressortent du diagnostic complet et doivent être reconnues dans toute utilisation opérationnelle :

**1. Sous-estimation systématique en queue haute** (Pilier 6)

```
Décile TARGET   % observations sous-estimées
D1 (calmes)        ~7%
D5                ~42%
D10 (volatils)    ~66%
```

Sur les jours de très forte volatilité, le modèle sous-estime 2 fois sur 3. C'est un effet de **shrinkage** classique des modèles ML — la régularisation tire les prédictions vers la moyenne. En risk management opérationnel, ce biais nécessiterait une recalibration multiplicative par décile ou une fonction de perte asymétrique.

**2. Performance dégradée sur les extrêmes**

```
Top 1% TARGET   : MAPE ≈ 24%
Top 0.5%        : MAPE ≈ 27%
Top 0.1%        : MAPE ≈ 30%
```

Le modèle perd en précision sur les vrais événements extrêmes, mais le niveau absolu reste acceptable. Une approche par Extreme Value Theory (EVT) pourrait améliorer ce comportement.

**3. Biais MAPE vers les petits TARGETs**

Le MAPE global de ~24% est mécaniquement tiré vers le haut par le quartile Q1 (TARGET faibles, MAPE ~36%) en raison de la nature relative de la métrique. C'est une **propriété de la métrique**, pas du modèle. Une métrique alternative comme QLIKE place le champion de manière cohérente.

## 9.4 Pistes pour améliorations futures

Les pistes ci-dessous n'ont pas été testées empiriquement dans ce projet. Elles sont documentées comme directions plausibles, sans garantie de gain.

### Piste 1 — Modèles séquentiels (LSTM, TCN, Transformer)

LightGBM traite les 54 barres comme des features indépendantes. Des architectures séquentielles pourraient exploiter la structure temporelle. **À tester** : TCN d'abord (compromis stabilité/performance), puis LSTM et Transformer.

**Trade-off à anticiper** : perte significative d'interprétabilité, complexité d'entraînement accrue.

### Piste 2 — Modèles classiques de séries temporelles (GARCH / EGARCH / HAR inter-day)

Modèles paramétriques de référence en volatilité (Bollerslev 1986, Nelson 1991, Corsi 2009). Leur application directe nécessite l'**ordre chronologique réel des dates**, anonymisé dans ce challenge. Pour les utiliser, il faudrait soit un dataset différent, soit un mapping vers un ordre temporel reconstitué.

### Piste 3 — Données externes

Le modèle actuel n'utilise que les données fournies par le challenge. Des données externes potentiellement utiles : VIX, returns S&P 500 intraday, volumes, indices sectoriels. **À évaluer** si l'accès à ces données est possible et si l'apport empirique justifie la complexité.

### Piste 4 — Calibration MAPE-aware par bucket

Calibration post-modèle adaptative : pour chaque décile de prédiction, apprendre un facteur multiplicatif sur OOF train. Le coût est modeste. **Trade-off identifié dans ce projet** : une expérimentation préliminaire avec `sample_weight = 1/y²` a montré un gain de MAPE global au prix d'une dégradation forte sur Q4. Une calibration par bucket plus fine pourrait préserver Q4. À tester rigoureusement avant intégration.

### Piste 5 — Ensembling de modèles structurellement diversifiés

Pour qu'un ensemble apporte du gain (contrairement à notre tentative HAR-RV + LightGBM), les modèles doivent avoir des **erreurs décorrélées**. Cela suppose des architectures fondamentalement différentes (par exemple LightGBM + TCN + GARCH avec données externes). L'effort marginal est conséquent et le gain reste à démontrer.